# Highway 97 Road Segment Congestion

This notebook downloads recent GTFS-Realtime vehicle and trip updates, runs the reusable congestion engine in `forecasting`, and visualizes segment-level congestion. Segments are consecutive stops along the representative Route 97 shape; a vehicle update is assigned to the segment ending at its `current_stop_sequence`.

## Requirements

Install the forecasting package and notebook dependencies from the repository (`cd forecasting && pip install -r requirements.txt`). Create the ignored `.env.road_segment_congestion` file by sourcing `./setup_road_segment_congestion_env.sh` from the repository root. This notebook loads the five `TRANSIT_DB_*` variables from that file automatically. Credentials are intentionally not stored in the notebook. The database pull defaults to the most recent three hours.

## Packages

In [ ]:
import os
from pathlib import Path
import shlex
import sys

import pandas as pd
import plotly.colors as pc
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from forecasting import download_backtest_data, run_congestion_engine

## Constants

In [ ]:


workspace_candidates = [Path.cwd().resolve(), Path.cwd().resolve().parent]
repo_root = next(
    (
        candidate
        for candidate in workspace_candidates
        if (candidate / 'data' / 'static_data').is_dir()
        and (candidate / 'forecasting' / 'forecasting').is_dir()
    ),
    None,
)
if repo_root is None:
    raise FileNotFoundError('Could not locate the repository data and forecasting package.')

env_file = repo_root / '.env.road_segment_congestion'
if not env_file.is_file():
    raise FileNotFoundError(
        f'{env_file} is missing. From the repository root, run '
        '`source ./setup_road_segment_congestion_env.sh` first.'
    )

required_env_vars = {
    'TRANSIT_DB_HOST',
    'TRANSIT_DB_USER',
    'TRANSIT_DB_PASSWORD',
    'TRANSIT_DB_NAME',
    'TRANSIT_DB_PORT',
}
for line_number, raw_line in enumerate(env_file.read_text().splitlines(), start=1):
    line = raw_line.strip()
    if not line or line.startswith('#'):
        continue
    if line.startswith('export '):
        line = line.removeprefix('export ').strip()
    variable, separator, encoded_value = line.partition('=')
    if not separator or variable not in required_env_vars:
        raise ValueError(f'Invalid environment setting on line {line_number} of {env_file}.')
    parsed_value = shlex.split(encoded_value, posix=True)
    if len(parsed_value) != 1:
        raise ValueError(f'Invalid value on line {line_number} of {env_file}.')
    os.environ[variable] = parsed_value[0]

missing_env_vars = sorted(required_env_vars - os.environ.keys())
if missing_env_vars:
    raise RuntimeError('Missing database settings: ' + ', '.join(missing_env_vars))



static_data_dir = repo_root / 'data' / 'static_data'

visuals_dir = repo_root / 'notebooks' / 'visuals'
visuals_dir.mkdir(parents=True, exist_ok=True)
print(f'Repository root: {repo_root}')

## Download backtest data

Both update tables are downloaded newest-first. The congestion calculations use vehicle updates; the trip updates are also returned for other backtest analyses.

In [ ]:
df_vehicle_updates, df_trip_updates = download_backtest_data(lookback_hours=24)
print(f'Vehicle updates: {len(df_vehicle_updates):,}')
print(f'Trip updates: {len(df_trip_updates):,}')
print(
    'Vehicle update window: '
    f'{df_vehicle_updates["feed_timestamp"].min()} through '
    f'{df_vehicle_updates["feed_timestamp"].max()}'
)

In [ ]:
df_stops = pd.read_csv(static_data_dir / 'stops.txt')
df_stop_times = pd.read_csv(static_data_dir / 'stop_times.txt')
df_trips = pd.read_csv(static_data_dir / 'trips.txt')
df_shapes = pd.read_csv(static_data_dir / 'shapes.txt')

print(
    f'Loaded {len(df_stops):,} stops, {len(df_stop_times):,} stop times, '
    f'{len(df_trips):,} trips, and {len(df_shapes):,} shape points.'
)

## Run the congestion engine

The package selects the Route 97 trip with the most stop-time records, slices its shape between consecutive stops, and matches updates to segments by `current_stop_sequence`. For each segment, the median of its historical probe speeds is the free-flow baseline. Mean speed is aggregated into five-minute UTC bins, and the congestion index is `1 - observed_speed / baseline_speed`, clipped to 0 (free flow) through 1 (stopped). Baselines at or below 0.5 are excluded because near-stationary layover speeds make the ratio unstable.

In [ ]:
route_id = '97-KEL'
time_bin_size = '5min'
congestion_result = run_congestion_engine(
    df_vehicle_updates,
    df_stops,
    df_stop_times,
    df_trips,
    df_shapes,
    route_id=route_id,
    time_bin_size=time_bin_size,
)

segments = congestion_result.segments
route_geometry = congestion_result.route_geometry
vehicle_probes = congestion_result.vehicle_probes
df_congestion = congestion_result.congestion
if df_congestion.empty:
    raise ValueError('No segment congestion estimates were produced for this data window.')

print(f'Representative trip: {congestion_result.representative_trip_id}')
print(f'Route segments: {len(segments)}')
print(f'Matched vehicle probes: {len(vehicle_probes):,}')
print(f'Congestion estimates: {len(df_congestion):,}')

## Latest segment congestion

Each line is colored using the most recent congestion estimate available for that segment. Gray indicates no reading in the selected window.

In [ ]:
latest_congestion = (
    df_congestion.sort_values('time_bin')
    .groupby('segment_id')
    .tail(1)
    .set_index('segment_id')['congestion_index']
)
colorscale = pc.get_colorscale('RdYlGn_r')

def congestion_to_color(value):
    if pd.isna(value):
        return 'gray'
    return pc.sample_colorscale(colorscale, [float(value)])[0]

fig = go.Figure()
for _, segment in segments.iterrows():
    segment_id = segment['segment_id']
    from_stop_name = segment['from_stop_name']
    to_stop_name = segment['to_stop_name']
    congestion = latest_congestion.get(segment_id, float('nan'))
    longitude, latitude = segment['geometry'].xy
    hover_text = (
        f'Segment {segment_id}: {from_stop_name} -> {to_stop_name}'
        + (
            f'<br>Congestion index: {congestion:.2f}'
            if pd.notna(congestion)
            else '<br>No recent data'
        )
    )
    fig.add_trace(
        go.Scattermap(
            lon=list(longitude),
            lat=list(latitude),
            mode='lines',
            line=dict(width=5, color=congestion_to_color(congestion)),
            name=f'{from_stop_name} -> {to_stop_name}',
            hovertext=hover_text,
            hoverinfo='text',
        )
    )

fig.update_layout(
    title=f'Route {route_id} Segment Congestion (latest reading)',
    map_style='open-street-map',
    map=dict(
        center=dict(lat=route_geometry.centroid.y, lon=route_geometry.centroid.x),
        zoom=11,
    ),
    showlegend=False,
    width=1500,
    height=700,
)
fig.write_html(visuals_dir / 'route_97_segment_congestion_engine.html')
fig.show()

## Spatial congestion animations

The following maps show the latest day in the downloaded data. Segment values are forward-filled between observations so a recent reading remains visible until it is replaced. The smoothed version uses a centered 10-bin rolling average for visualization; it is not a real-time causal estimate.

In [ ]:
animation_window = pd.Timedelta(days=1)
max_time_bin = df_congestion['time_bin'].max()
df_animation = df_congestion.loc[
    df_congestion['time_bin'] >= max_time_bin - animation_window
].copy()

raw_congestion_grid = (
    df_animation.pivot(
        index='time_bin',
        columns='segment_id',
        values='congestion_index',
    )
    .sort_index()
    .reindex(columns=segments['segment_id'])
    .ffill()
)
smoothed_congestion_grid = raw_congestion_grid.rolling(
    window=10,
    min_periods=1,
    center=True,
).mean()

def make_congestion_animation(congestion_grid, title, value_label):
    if congestion_grid.empty:
        raise ValueError('No congestion data is available to animate.')

    frames = []
    for time_bin, row_values in congestion_grid.iterrows():
        frame_data = []
        for _, segment in segments.iterrows():
            segment_id = segment['segment_id']
            from_stop_name = segment['from_stop_name']
            to_stop_name = segment['to_stop_name']
            congestion = row_values[segment_id]
            longitude, latitude = segment['geometry'].xy
            hover_text = (
                f'Segment {segment_id}: {from_stop_name} -> {to_stop_name}'
                + (
                    f'<br>{value_label}: {congestion:.2f}'
                    if pd.notna(congestion)
                    else '<br>No data yet'
                )
            )
            frame_data.append(
                go.Scattermap(
                    lon=list(longitude),
                    lat=list(latitude),
                    mode='lines',
                    line=dict(
                        width=5,
                        color=congestion_to_color(congestion),
                    ),
                    name=f'{from_stop_name} -> {to_stop_name}',
                    hovertext=hover_text,
                    hoverinfo='text',
                )
            )
        frames.append(go.Frame(name=str(time_bin), data=frame_data))

    figure = go.Figure(data=frames[0].data, frames=frames)
    figure.update_layout(
        title=title,
        map_style='open-street-map',
        map=dict(
            center=dict(
                lat=route_geometry.centroid.y,
                lon=route_geometry.centroid.x,
            ),
            zoom=11,
        ),
        showlegend=False,
        width=1500,
        height=700,
        updatemenus=[
            {
                'type': 'buttons',
                'showactive': False,
                'x': 0.0,
                'y': 1.05,
                'buttons': [
                    {
                        'label': 'Play',
                        'method': 'animate',
                        'args': [
                            None,
                            {
                                'frame': {'duration': 250, 'redraw': True},
                                'fromcurrent': True,
                                'transition': {'duration': 0},
                            },
                        ],
                    },
                    {
                        'label': 'Pause',
                        'method': 'animate',
                        'args': [
                            [None],
                            {
                                'frame': {'duration': 0, 'redraw': True},
                                'mode': 'immediate',
                                'transition': {'duration': 0},
                            },
                        ],
                    },
                ],
            }
        ],
        sliders=[
            {
                'x': 0.08,
                'len': 0.9,
                'currentvalue': {'prefix': 'Time bin (UTC): '},
                'steps': [
                    {
                        'label': pd.Timestamp(frame.name).strftime('%m-%d %H:%M'),
                        'method': 'animate',
                        'args': [
                            [frame.name],
                            {
                                'frame': {'duration': 0, 'redraw': True},
                                'mode': 'immediate',
                                'transition': {'duration': 0},
                            },
                        ],
                    }
                    for frame in frames
                ],
            }
        ],
    )
    return figure

In [ ]:
raw_animation = make_congestion_animation(
    raw_congestion_grid,
    title=f'Route {route_id} Segment Congestion Over Time (last {animation_window})',
    value_label='Congestion index',
)
raw_animation.write_html(
    visuals_dir / 'route_97_segment_congestion_engine_animation.html'
)
raw_animation.show()

In [ ]:
smoothed_animation = make_congestion_animation(
    smoothed_congestion_grid,
    title=(
        f'Route {route_id} Smoothed Segment Congestion Over Time '
        f'(last {animation_window}, 10-bin rolling mean)'
    ),
    value_label='Smoothed congestion index',
)
smoothed_animation.write_html(
    visuals_dir / 'route_97_smoothed_segment_congestion_engine_animation.html'
)
smoothed_animation.show()

## Segment time series and probe availability

These time series show raw and five-bin rolling-mean congestion for segment 6, plus the number of distinct buses contributing in each bin. Timestamps are converted from UTC to Vancouver local time for display.

In [ ]:
segment_id_of_interest = 6
segment_matches = segments.loc[
    segments['segment_id'].eq(segment_id_of_interest)
]
if segment_matches.empty:
    raise ValueError(f'Segment {segment_id_of_interest} is not present on route {route_id}.')
segment = segment_matches.iloc[0]
segment_label = f"{segment['from_stop_name']} -> {segment['to_stop_name']}"

df_single_segment = (
    df_congestion.loc[
        df_congestion['segment_id'].eq(segment_id_of_interest)
    ]
    .sort_values('time_bin')
    .copy()
)
df_single_segment['time_bin_local'] = df_single_segment['time_bin'].dt.tz_convert(
    'America/Vancouver'
)
segment_smoothing_window = 5
df_single_segment['congestion_index_smooth'] = (
    df_single_segment['congestion_index']
    .rolling(
        window=segment_smoothing_window,
        min_periods=1,
        center=True,
    )
    .mean()
)

segment_figure = go.Figure()
segment_figure.add_trace(
    go.Scatter(
        x=df_single_segment['time_bin_local'],
        y=df_single_segment['congestion_index'],
        mode='lines+markers',
        name='Raw',
        line=dict(color='lightgray'),
        opacity=0.6,
    )
)
segment_figure.add_trace(
    go.Scatter(
        x=df_single_segment['time_bin_local'],
        y=df_single_segment['congestion_index_smooth'],
        mode='lines+markers',
        name=f'{segment_smoothing_window}-bin rolling mean',
        line=dict(color='darkorange'),
    )
)
segment_figure.update_layout(
    title=f'Smoothed Congestion Index for Segment: {segment_label}',
    xaxis_title='Time (Vancouver)',
    yaxis_title='Congestion Index (0 = free flow, 1 = stopped)',
)
segment_figure.show()

In [ ]:
df_vehicle_updates

In [ ]:
df_probe_counts = (
    vehicle_probes.loc[
        vehicle_probes['segment_id'].eq(segment_id_of_interest)
    ]
    .groupby('time_bin')['vehicle_id']
    .nunique()
    .rename('probe_count')
    .reset_index()
    .sort_values('time_bin')
)
df_probe_counts['time_bin_local'] = df_probe_counts['time_bin'].dt.tz_convert(
    'America/Vancouver'
)

probe_figure = make_subplots(
    rows=2,
    cols=1,
    shared_xaxes=True,
    row_heights=[0.6, 0.4],
    vertical_spacing=0.08,
    subplot_titles=(
        f'Congestion Index for Segment: {segment_label}',
        'Number of Buses (Probes) Used per Time Bin',
    ),
)
probe_figure.add_trace(
    go.Scatter(
        x=df_single_segment['time_bin_local'],
        y=df_single_segment['congestion_index'],
        mode='lines+markers',
        name='Raw',
        line=dict(color='lightgray'),
        opacity=0.6,
    ),
    row=1,
    col=1,
)
probe_figure.add_trace(
    go.Scatter(
        x=df_single_segment['time_bin_local'],
        y=df_single_segment['congestion_index_smooth'],
        mode='lines+markers',
        name=f'{segment_smoothing_window}-bin rolling mean',
        line=dict(color='darkorange'),
    ),
    row=1,
    col=1,
)
probe_figure.add_trace(
    go.Bar(
        x=df_probe_counts['time_bin_local'],
        y=df_probe_counts['probe_count'],
        name='Probe count',
        marker=dict(color='steelblue'),
    ),
    row=2,
    col=1,
)
probe_figure.update_yaxes(
    title_text='Congestion Index (0 = free flow, 1 = stopped)',
    row=1,
    col=1,
)
probe_figure.update_yaxes(title_text='# Buses', row=2, col=1)
probe_figure.update_xaxes(title_text='Time (Vancouver)', row=2, col=1)
probe_figure.update_layout(
    title=f'Congestion Index vs. Probe Availability: {segment_label}',
    showlegend=True,
    height=700,
)
probe_figure.show()